# AI-assisted workflow

This notebook repeats the workflow of this week’s lesson, but this time an
AI assistant helps us write the code. The division of labour is simple:

- **You decide the GIS part**: which data, which coordinate reference system,
  which spatial predicate, which classes. You are the GIS expert, and these
  decisions go *into* your prompt.
- **The AI assistant helps with the Python part**: boilerplate, syntax you
  don’t remember, restructuring code, explaining error messages.

The responses shown below are *examples*. If you run the same prompts yourself,
you will get different code, and that is fine. What matters is that you can
read the code, run it, and check the result.

This week, we use an AI assistant to automate the reading and writing of the
topographic database files we explored in the lesson.

> **Habit of the week: you decide, the AI writes the code**
>
> A prompt should contain *your* decisions: file names, the coordinate
> reference system, the output format. The more precise your decisions, the
> better the code you get back. Also tell the assistant which library versions
> you use: libraries change, and code written for an older version might not
> work in this year’s environment.

## Your decisions

- Input: all polygon layers (`*_p.shp`) of the topographic database in `data/finland_topographic_database/`.
- Coordinate reference system: **ETRS-TM35FIN (EPSG:3067)**, the national CRS of Finland, which uses metres. Any layer in a different CRS is reprojected.
- Output: one *GeoPackage*, one layer per input file, named after the input file.
- Areas are reported in km².

In [ ]:
import pathlib
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
TOPOGRAPHIC_DATABASE_DIRECTORY = DATA_DIRECTORY / "finland_topographic_database"

## Step 0: collect the context for your prompt

Find out which versions of the main libraries are installed, so you can
mention them in the prompt:

In [ ]:
import geopandas
import pandas
import shapely

print(f"geopandas {geopandas.__version__}, pandas {pandas.__version__}, shapely {shapely.__version__}")

## Step 1: many shapefiles into one GeoPackage

**Prompt:**

```text
I use geopandas 1.x and pathlib. Loop over all files matching "*_p.shp" in the
folder TOPOGRAPHIC_DATABASE_DIRECTORY (a pathlib.Path). For each file: read it,
reproject it to EPSG:3067 if its CRS is different, and write it as a layer into
one GeoPackage "topographic_database.gpkg" in DATA_DIRECTORY. Use the file name
without extension as the layer name. Print the layer name and the number of
rows.
```

**Example response** (yours may differ):

In [ ]:
import geopandas

OUTPUT_FILE = DATA_DIRECTORY / "topographic_database.gpkg"
TARGET_CRS = "EPSG:3067"

row_counts = {}

for input_file in sorted(TOPOGRAPHIC_DATABASE_DIRECTORY.glob("*_p.shp")):
    layer = geopandas.read_file(input_file)

    if layer.crs != TARGET_CRS:
        layer = layer.to_crs(TARGET_CRS)

    layer.to_file(OUTPUT_FILE, layer=input_file.stem, driver="GPKG")
    row_counts[input_file.stem] = len(layer)
    print(f"{input_file.stem}: {len(layer)} rows")

**Before you run it**, read it through:

- `sorted()` makes sure the files are always processed in the same order.
- `input_file.stem` is the file name without its extension (see [Managing
  file paths](managing-file-paths)).
- If a layer with the same name already exists in the GeoPackage, `to_file()`
  overwrites it. That’s what we want here, as we might run the notebook several
  times.
- The code remembers the number of rows per layer in `row_counts`, which we
  can use to check the result.

## Check the result

Read the GeoPackage back and compare it with what we wrote:

In [ ]:
layers = geopandas.list_layers(OUTPUT_FILE)
layers

In [ ]:
for layer_name, expected_rows in row_counts.items():
    layer = geopandas.read_file(OUTPUT_FILE, layer=layer_name)
    assert len(layer) == expected_rows, f"{layer_name}: row count differs"
    assert layer.crs == TARGET_CRS, f"{layer_name}: CRS differs"

print(f"All {len(row_counts)} layers have the expected number of rows and CRS {TARGET_CRS}.")

## Step 2: a tidy summary table

In the lesson, we calculated the total area per terrain class with
`groupby()`. Let’s ask for a more complete summary table:

**Prompt:**

```text
In geopandas, I have a GeoDataFrame `terrain` (CRS EPSG:3067, unit metres) with
a column "LUOKKA" (terrain class). Create a summary table with one row per
class and the columns "class", "polygons" (number of polygons) and "area_km2"
(total area in km²), sorted by area, largest first.
```

**Example response** (yours may differ):

In [ ]:
terrain = geopandas.read_file(OUTPUT_FILE, layer="m_L4132R_p")

area_by_class = (
    terrain
    .assign(area_km2=terrain.area / 1_000_000)
    .groupby("LUOKKA")
    .agg(
        polygons=("area_km2", "count"),
        area_km2=("area_km2", "sum"),
    )
    .sort_values("area_km2", ascending=False)
    .rename_axis("class")
    .reset_index()
)
area_by_class

**Before you run it**: this is a *method chain*. Read it top to bottom, one
line at a time: add an area column (in km², because the CRS unit is metres),
group by class, count and sum, sort, rename the index, turn the index back
into a column. If a chain is hard to follow, run it step by step, adding one
line at a time.

## Check the result

We know some numbers from the lesson: there are 56 lake polygons (class
`36200`), and the total area cannot change by grouping:

In [ ]:
lakes = area_by_class[area_by_class["class"] == 36200]
assert lakes["polygons"].item() == 56

assert abs(area_by_class["area_km2"].sum() - terrain.area.sum() / 1_000_000) < 1e-6
assert area_by_class["polygons"].sum() == len(terrain)

print("The summary table matches the lesson.")

## AI-use log

Whenever you use an AI assistant in this course (in exercises, only where a
task is marked **AI-LLM OK**, see the [course guidelines on AI
tools](https://autogis-site.readthedocs.io/en/latest/course-info/ai-tools.html)),
keep a short log of what you did. For this notebook, it could look like this:

| Step | What I asked | What I changed | How I checked it |
|------|--------------|----------------|------------------|
| 1 | A loop that writes all polygon shapefiles into one GeoPackage, in EPSG:3067 | Nothing | Read all layers back, compared row counts and CRS |
| 2 | A summary table of polygons and area (km²) per terrain class | Nothing | Compared with the lesson (56 lakes), checked that totals are unchanged |